In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — immediately after Drive mount. Safe for a brand-new Runtime → Run all.
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_ROOT=DRIVE_ROOT + '/LCX_Curved_Template_Reacquisition_v1'
BRANCH='lcx-curved-template-reacquisition-from-main'


# OpenPlaque — LCX curved-template source reacquisition

The historical Series-1039 LCX-labeled mask is treated only as an **unlabeled coronary template**. The stale historical LCX source centerline is never used. Candidate paths come from current+legacy TotalSegmentator source-space coronary anatomy. A matched path is not called LCX until a later independent anatomical/topological validation succeeds.


In [ ]:
import os, shutil, subprocess, sys
shutil.rmtree('/content/OpenPlaque', ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git','/content/OpenPlaque'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-image'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','/content/OpenPlaque'], check=True)
subprocess.run([sys.executable,'-c','import openplaque; print(openplaque.__file__)'], check=True)


In [ ]:
# Fail clearly on source corruption before running pytest.
subprocess.run([sys.executable,'-m','py_compile','/content/OpenPlaque/src/openplaque/lcx_curved_template_reacquisition.py'], check=True)
subprocess.run([sys.executable,'-m','pytest','-q','/content/OpenPlaque/tests/test_lcx_curved_template_reacquisition.py'], check=True)


In [ ]:
# Run the actual workflow in a fresh Python subprocess too.
# This avoids relying on the already-running notebook kernel to discover
# the editable install's newly-created .pth/import hook.
runner = f'''
from openplaque.lcx_curved_template_reacquisition import synthetic_lcx_template_self_test, run
print('SELF TEST:', synthetic_lcx_template_self_test())
result = run({DRIVE_ROOT!r}, {OUTPUT_ROOT!r})
print('STATUS:', result['summary']['status'])
print('REPORT:', result['report'])
print('ZIP:', result['zip'])
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT_BACK.zip')
'''
subprocess.run([sys.executable,'-c',runner], check=True)
